In [1]:
import requests
import time
import json
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
endpoint = 'https://data.cityofnewyork.us/resource/h9gi-nx95.json'
target_rows = 1_000_000  # processar apenas 1 milhão de linhas
pagesize = 500000
out_raw = 'data.csv'
download_log = 'logs.json'
delay_between_requests = 0.2  # segundos; aumente se precisar reduzir taxa

# Estado simples do download
offset = 0
first_chunk = True
agg_log = {'chunks': 0, 'rows_downloaded': 0, 'errors': []}

print(f'Iniciando download paginado (target={target_rows} linhas)...')
while agg_log['rows_downloaded'] < target_rows:
    params = {'$limit': pagesize, '$offset': offset}
    try:
        r = requests.get(endpoint, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
    except Exception as e:
        msg = f'Erro na requisição offset={offset}: {e}'
        print(msg)
        agg_log['errors'].append(msg)
        break

    if not data:
        print('Nenhum registro retornado — fim do dataset (offset=', offset, ')')
        break

    df_chunk = pd.DataFrame(data)

    # Se exceder target_rows, truncar o chunk
    remaining = target_rows - agg_log['rows_downloaded']
    if len(df_chunk) > remaining:
        df_chunk = df_chunk.iloc[:remaining]

    # Gravando RAW em CSV (apende)
    header = first_chunk
    df_chunk.to_csv(out_raw, mode='a', index=False, header=header, encoding='utf-8')
    first_chunk = False

    # Atualiza agregados simples
    agg_log['chunks'] += 1
    agg_log['rows_downloaded'] += len(df_chunk)

    print(f"Chunk {agg_log['chunks']} rows={len(df_chunk)} appended (offset={offset})")
    offset += pagesize
    time.sleep(delay_between_requests)

# salvar log de download
with open(download_log, 'w', encoding='utf-8') as f:
    json.dump(agg_log, f, indent=2, ensure_ascii=False)

print('Download concluído. RAW salvo em:', out_raw)
print('Log de download salvo em:', download_log)

Iniciando download paginado (target=1000000 linhas)...
Chunk 1 rows=500000 appended (offset=0)
Chunk 2 rows=500000 appended (offset=500000)
Download concluído. RAW salvo em: data.csv
Log de download salvo em: logs.json


In [3]:
# Célula 2: Limpeza — ler o RAW (`raw_1M.csv`) em chunks e gravar `cleaned_1M.csv`
# Esta célula aplica as transformações previamente presentes na célula única original.
import json
from pathlib import Path
import pandas as pd
import numpy as np
import data_cleaning

in_csv = 'data.csv'
out_csv = 'cleaned_data.csv'
log_path = 'cleaned_logs.json'
pagesize_read = 500000  # ajuste conforme memória

first_chunk = True
agg_log = {'chunks': 0, 'rows_processed': 0, 'per_step_counts': {}, 'errors': []}

print(f'Read/clean pipeline: lendo {in_csv} em chunks de {pagesize_read}...')
for df_chunk in pd.read_csv(in_csv, chunksize=pagesize_read, dtype=str, keep_default_na=False):
    # normalizar nomes de colunas (se desejado)
    df_chunk = data_cleaning.normalize_colnames(df_chunk)

    # truncamento por segurança (se arquivo raw for maior que o previsto)
    remaining = 1_000_000 - agg_log['rows_processed']
    if len(df_chunk) > remaining:
        df_chunk = df_chunk.iloc[:remaining]

    chunk_stats = {'rows': len(df_chunk)}

    # 1) Colunas de contagem: coerção e imputação 0 (contabilizar mudanças)
    # Implementação defensiva: tenta coagir para numérico, faz limpeza adicional se necessário,
    # e usa pandas nullable Int64 quando possível; caso contrário, faz fallback para float.
    def _safe_int64_coerce(series):
        # preserva a série original (não sobrescrevemos aqui) e retorna um array/serie coerente
        s = series.replace('', np.nan)
        # 1) tentativa direta com to_numeric
        try:
            s_num = pd.to_numeric(s, errors='coerce')
        except Exception:
            # se houver entradas não-escalares (listas/dicts) ou outro tipo problemático,
            # convertemos para str e removemos caracteres não-numéricos antes de coagir
            s_str = s.astype(str).str.replace(r'[^0-9]', '', regex=True)
            s_str = s_str.replace('', np.nan)
            s_num = pd.to_numeric(s_str, errors='coerce')
        # 2) agora temos uma série numérica (ou NaN); tentar converter para Int64 nullable
        try:
            int_arr = pd.array(s_num.fillna(0), dtype='Int64')
            return int_arr, int(s_num.isna().sum())
        except Exception:
            # tentativa alternativa usando astype (algumas versões de pandas diferem em comportamento)
            try:
                int_ser = s_num.fillna(0).astype('Int64')
                return int_ser, int(s_num.isna().sum())
            except Exception:
                # fallback final: manter como float (valores decimais esperados) e retornar contagem de NA
                float_ser = s_num.fillna(0).astype('float')
                return float_ser, int(s_num.isna().sum())

    count_cols = [c for c in df_chunk.columns if data_cleaning.is_count_col(c)]
    chunk_stats['count_cols'] = count_cols
    chunk_stats['counts'] = {}
    for c in count_cols:
        before_missing = int(df_chunk[c].replace('', np.nan).isna().sum()) if c in df_chunk.columns else 0
        if c in df_chunk.columns:
            coerced, after_missing = _safe_int64_coerce(df_chunk[c])
            # coerced é ou um ExtensionArray/Series compatível (Int64) ou float series
            df_chunk.loc[:, c] = coerced
        else:
            after_missing = 0
        chunk_stats['counts'][c] = {'missing_before': before_missing, 'missing_after': after_missing}

    # 2) Datas/horas (leves coerções)
    if 'crash_date' in df_chunk.columns:
        df_chunk.loc[:, 'crash_date_parsed'] = pd.to_datetime(df_chunk['crash_date'], errors='coerce').dt.date
    if 'crash_time' in df_chunk.columns:
        def _parse_time(x):
            try:
                if pd.isna(x) or str(x).strip() == '':
                    return pd.NaT
                t = pd.to_datetime(str(x).strip(), format='%H:%M', errors='coerce')
                if pd.isna(t):
                    t = pd.to_datetime(str(x).strip(), format='%H:%M:%S', errors='coerce')
                if pd.isna(t):
                    t = pd.to_datetime(str(x).strip(), errors='coerce')
                return t.time() if not pd.isna(t) else pd.NaT
            except Exception:
                return pd.NaT
        df_chunk.loc[:, 'crash_time_parsed'] = df_chunk['crash_time'].apply(_parse_time)

    # 3) Latitude/Longitude: coerção e zeros->NaN
    for loc in ('latitude','longitude'):
        if loc in df_chunk.columns:
            before_zero = int(pd.to_numeric(df_chunk[loc], errors='coerce').fillna(np.nan).eq(0).sum())
            df_chunk.loc[:, loc] = pd.to_numeric(df_chunk[loc].replace('', np.nan), errors='coerce')
            df_chunk.loc[df_chunk[loc] == 0.0, loc] = np.nan
            chunk_stats.setdefault('latlon', {})[loc] = {'zeros_before': before_zero, 'n_missing_after': int(df_chunk[loc].isna().sum())}

    # 4) zip code and borough standardization
    if 'zip_code' in df_chunk.columns:
        df_chunk.loc[:, 'zip_code_std'] = df_chunk['zip_code'].apply(data_cleaning.standardize_zip)
    if 'borough' in df_chunk.columns:
        df_chunk.loc[:, 'borough_std'] = df_chunk['borough'].astype(str).str.strip().str.upper().replace({'NONE': None, '': None})

    # 5) inconsistências: contar e aplicar correção
    # injured
    if 'number_of_persons_injured' in df_chunk.columns:
        comp_inj_cols = [c for c in df_chunk.columns if c.endswith('_injured') and c != 'number_of_persons_injured']
        if comp_inj_cols:
            comp_sum = df_chunk[comp_inj_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
            total = pd.to_numeric(df_chunk['number_of_persons_injured'], errors='coerce').fillna(0)
            mask = comp_sum > total
            chunk_stats.setdefault('inconsistencies', {})['injured_before_fix'] = int(mask.sum())
    # killed
    if 'number_of_persons_killed' in df_chunk.columns:
        comp_kill_cols = [c for c in df_chunk.columns if c.endswith('_killed') and c != 'number_of_persons_killed']
        if comp_kill_cols:
            comp_sum_k = df_chunk[comp_kill_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
            total_k = pd.to_numeric(df_chunk['number_of_persons_killed'], errors='coerce').fillna(0)
            maskk = comp_sum_k > total_k
            chunk_stats.setdefault('inconsistencies', {})['killed_before_fix'] = int(maskk.sum())

    # apply fix_inconsistencies (function will update df_chunk and may return counts in its log)
    df_chunk = data_cleaning.fix_inconsistencies(df_chunk, chunk_stats)

    # after-fix masks (recompute)
    if 'number_of_persons_injured' in df_chunk.columns and 'injured_before_fix' in chunk_stats.get('inconsistencies', {}):
        comp_inj_cols = [c for c in df_chunk.columns if c.endswith('_injured') and c != 'number_of_persons_injured']
        if comp_inj_cols:
            comp_sum = df_chunk[comp_inj_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
            total = pd.to_numeric(df_chunk['number_of_persons_injured'], errors='coerce').fillna(0)
            mask_after = comp_sum > total
            chunk_stats['inconsistencies']['injured_after_fix'] = int(mask_after.sum())
    if 'number_of_persons_killed' in df_chunk.columns and 'killed_before_fix' in chunk_stats.get('inconsistencies', {}):
        comp_kill_cols = [c for c in df_chunk.columns if c.endswith('_killed') and c != 'number_of_persons_killed']
        if comp_kill_cols:
            comp_sum_k = df_chunk[comp_kill_cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
            total_k = pd.to_numeric(df_chunk['number_of_persons_killed'], errors='coerce').fillna(0)
            maskk_after = comp_sum_k > total_k
            chunk_stats['inconsistencies']['killed_after_fix'] = int(maskk_after.sum())

    # 6) Outliers: detectar e aplicar winsorization por chunk (com flags)
    outlier_log = {}
    # Para cada coluna de contagem, marcar outliers (1%/99%) e fazer capping (winsorize).
    for c in count_cols:
        try:
            # garantir série numérica (já usamos Int64 acima, mas reforçar)
            ser = pd.to_numeric(df_chunk[c], errors='coerce').astype('float')
            if ser.dropna().empty:
                outlier_log[c] = {'n_outliers': 0, 'lower': None, 'upper': None}
                continue
            lower = float(ser.quantile(0.01))
            upper = float(ser.quantile(0.99))
            # mask de outliers antes do capping
            mask_low = ser < lower
            mask_high = ser > upper
            mask = mask_low | mask_high
            n_out = int(mask.sum())
            # flag de outlier e cópia do valor original (apenas para outliers)
            if n_out > 0:
                df_chunk.loc[mask, f'{c}_is_outlier'] = True
                # manter o original para auditoria
                df_chunk.loc[mask, f'{c}_outlier_orig'] = df_chunk.loc[mask, c]
            else:
                # garantir coluna existindo com False quando não há outliers
                df_chunk.loc[:, f'{c}_is_outlier'] = df_chunk.get(f'{c}_is_outlier', False)
            # aplicar winsorization (capping) onde aplicável
            df_chunk.loc[mask_high, c] = upper
            df_chunk.loc[mask_low, c] = lower
            outlier_log[c] = {'n_outliers': n_out, 'lower': lower, 'upper': upper}
        except Exception as e:
            outlier_log[c] = {'error': str(e)}

    # Permitir que a função externa faça tratamentos adicionais (se implementada)
    try:
        df_chunk = data_cleaning.detect_and_treat_outliers(df_chunk, count_cols, outlier_log)
    except Exception:
        # se a função externa não existir ou falhar, continuamos com o que fizemos
        pass

    chunk_stats['outliers'] = outlier_log

    # Append to CSV (header only for first chunk)
    header = first_chunk
    df_chunk.to_csv(out_csv, mode='a', index=False, header=header, encoding='utf-8')
    first_chunk = False

    # aggregate stats
    agg_log['chunks'] += 1
    agg_log['rows_processed'] += chunk_stats['rows']
    # merge per_step_counts (sum counters)
    for k,v in chunk_stats.items():
        if k == 'rows':
            continue
        if k not in agg_log['per_step_counts']:
            agg_log['per_step_counts'][k] = v
        else:
            # naive merge for dicts: if both dicts, update keys by summing where numeric
            if isinstance(v, dict) and isinstance(agg_log['per_step_counts'][k], dict):
                for kk,vv in v.items():
                    if kk not in agg_log['per_step_counts'][k]:
                        agg_log['per_step_counts'][k][kk] = vv
                    else:
                        try:
                            agg_log['per_step_counts'][k][kk] = int(agg_log['per_step_counts'][k][kk]) + int(vv)
                        except Exception:
                            agg_log['per_step_counts'][k][kk] = vv
            else:
                agg_log['per_step_counts'][k] = v

    print(f"Chunk {agg_log['chunks']} rows={chunk_stats['rows']} cleaned and appended")

# finalizar e gravar log agregado
with open(log_path, 'w', encoding='utf-8') as f:
    json.dump(agg_log, f, indent=2, ensure_ascii=False)

print('Processamento de limpeza concluído. Arquivo limpo:', out_csv)
print('Log salvo em:', log_path)

Read/clean pipeline: lendo data.csv em chunks de 500000...


/root/data_science/data_cleaning.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  comp_sum = df[comp_inj_cols].fillna(0).sum(axis=1)
/root/data_science/data_cleaning.py:105: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  comp_sum = df[comp_kill_cols].fillna(0).sum(axis=1)


Chunk 1 rows=500000 cleaned and appended


/root/data_science/data_cleaning.py:96: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  comp_sum = df[comp_inj_cols].fillna(0).sum(axis=1)
/root/data_science/data_cleaning.py:105: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  comp_sum = df[comp_kill_cols].fillna(0).sum(axis=1)


Chunk 2 rows=500000 cleaned and appended
Processamento de limpeza concluído. Arquivo limpo: cleaned_data.csv
Log salvo em: cleaned_logs.json
